# 📖 تلاوة — تشغيل الأداة على Google Colab

1. (اختياري، للسرعة) من القائمة: **Runtime ← Change runtime type ← T4 GPU**.
2. شغّل الخلية التالية (▶). تأخذ 3–5 دقائق أول مرة.
3. اضغط الرابط الذي يظهر في آخرها: تفتح الأداة في صفحة جديدة.

الجلسة تُغلق بعد مدة من عدم الاستخدام؛ عندها شغّل الخلية من جديد. حمّل الفيديو من زر التحميل في الأداة قبل الإغلاق.

In [ ]:
#@title ▶ تشغيل الأداة
import os, subprocess, time, urllib.request
BRANCH = "ccr-d73776b9-i8u1ia"
if not os.path.exists("/content/repo"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, "--filter=blob:none", "--sparse",
                    "https://github.com/mytatrayd-boop/-.git", "/content/repo"], check=True)
    subprocess.run(["git", "-C", "/content/repo", "sparse-checkout", "set", "apps/tilawa"], check=True)
else:
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
APP = "/content/repo/apps/tilawa"
print("① تثبيت المكتبات…")
subprocess.run("apt-get -qq install -y ffmpeg libfribidi0 libraqm0 > /dev/null", shell=True)
subprocess.run("pip -q install -U streamlit transformers pillow pandas", shell=True, check=True)
print("② نص المصحف والخطوط…")
subprocess.run(["python", "setup_assets.py"], cwd=APP, check=True)
print("③ تشغيل الأداة…")
subprocess.run("pkill -f 'streamlit run' || true", shell=True)
subprocess.Popen("streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > /content/tilawa.log 2>&1",
                 shell=True, cwd=APP)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:8501", timeout=2); break
    except Exception:
        time.sleep(2)
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
import torch
print("كرت الشاشة:", "يعمل ✅" if torch.cuda.is_available() else "غير مفعّل (المزامنة أبطأ)")
from IPython.display import HTML, display
display(HTML(f'<h2 dir="rtl"><a href="{url}" target="_blank">⬅️ افتح أداة تلاوة</a></h2>'))
